In [22]:
from Bio import Entrez
from Bio import SeqIO
from Bio.SeqIO import read
from Bio.Seq import MutableSeq, Seq
from Bio.SeqRecord import SeqRecord
import os 
import random
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.figure_factory as ff
import plotly.graph_objects as go
import plotly.express as px
import pyorthoani
import itertools
from scipy.stats import linregress
from watermark import watermark
import kaleido
from collections import defaultdict
import multiprocessing
from dataclasses import dataclass
from typing import Generator, Callable
from copy import copy,deepcopy
import heapq
import subprocess

In [10]:
dependecies=watermark(iversions=True,globals_=globals(),python=True,conda=True)
dependecies=dependecies.split('\n')
for i,dep in enumerate(dependecies):
    dependecies[i]=dep.replace(' ','').replace(':','==')
with open('requirements.txt','w') as f:
    for dep in dependecies:
        f.write(dep+'\n')    

In [11]:
outdir= "NCBI_downloads"
os.makedirs(outdir,exist_ok=True)

Entrez.email = 'lorenzo.barletta@studenti.unipd.it'

Downloadin sequences from NCBI 

In [12]:
seq_record_str=[]
seq_record= []
# Define a list of genome IDs
genome_ids = ['U00096.3', 'BA000007.3', 'CP002967.1']

# Create the directory if it doesn't exist
if not os.path.exists(outdir):
    os.makedirs(outdir)
# Download genome sequences for IDs whose files are not present in the directory
for genome_id in genome_ids:
    filename = os.path.join(outdir, f"{genome_id}.fasta")  # Specify the output directory in the filename

    if not os.path.isfile(filename):
        try:
            net_handle = Entrez.efetch(db='nucleotide', id=genome_id, rettype='fasta', retmode='text')
            with open(filename, "w") as out_handle: 
                out_handle.write(net_handle.read())
            net_handle.close()
        except Exception as e:
            print(f"Error downloading genome {genome_id}: {e}")

for genome_id in genome_ids:
    file = f"{genome_id}.fasta"
    for record in SeqIO.parse(os.path.join(outdir, file), 'fasta'): 
        seq_record.append((record.id,record.seq))
        seq_record_str.append(str(record.seq))


SNP

In [4]:
mut_upto1 = np.linspace(0.001,0.01,10)
mut_upto10 = np.linspace(0.02,0.1,9)
mut_array= np.concatenate((mut_upto1,mut_upto10))

In [5]:
basiupper=["A","T","C","G"]
weights=[0.25,0.25,0.25,0.25]

In [6]:
def make_synth_seq(N: int,seq_length: int) -> Generator[MutableSeq, None, None]:
    for i in range(N):
        sequence = MutableSeq("".join(random.choice(basiupper)for i in range(seq_length)))
        yield sequence

## SNP function optimized 
Takes a single sequence as input and applies the mutation rate
The sequence is a ` MutableSeq ` which allows for inplace replacement of characters without needing to convert to a ` list ` object 

In [7]:
@dataclass(slots=True)
class Mutation:
    index: int
    mutation_type: str

@dataclass(slots=True)
class GenomeMutations:
    indices: list[Mutation]
    number_of_mutations: int

def snp_1_opt(sequence: MutableSeq, mutation_rate: float, psrandom: bool) -> tuple[MutableSeq, GenomeMutations]:
    real_mutation_rate = mutation_rate * 4/3
    number_of_mutations = int(real_mutation_rate * len(sequence))
    mutations = GenomeMutations([], 0)
    mutated_sequence = deepcopy(sequence)
    if psrandom == True:            # TODO I COMPITI PER CASA xxxo
        
            # Select 3 random splits of the sequence indexes to mutate 
            index_range = np.arange(len(sequence))
            split_index = np.array_split(index_range, 10) # Split the indexes into 10 parts
            selected_index = random.sample(split_index,3) # Select 3 random splits
            joined_selected_index = np.concatenate(selected_index)
            
            indexes = np.random.choice(joined_selected_index,size=number_of_mutations,replace=False)

    else:
        indexes = np.random.choice(len(sequence),size=number_of_mutations,replace=False)

    for i in indexes:
        replaced_base = sequence[i]
        new_base = random.choice(basiupper)
        if new_base != replaced_base:
            mutated_sequence[i] = new_base
            mutations.indices.append(Mutation(i, "snp"))

    mutations.number_of_mutations = len(mutations.indices)
    return mutated_sequence, mutations

In [17]:

def generate_kde_fig(mutation_indexes: GenomeMutations, seq_no: int, mut_rate: float, color):
    kde_fig = ff.create_distplot(
        [[x.index for x in mutation_indexes.indices]], 
        [f'{seq_no}[{round(mut_rate*100, 2)}]'], 
        colors=[color], 
        show_hist=False, 
        show_rug=False
    )
    return kde_fig

def plotly_kde_dropdown(kde_figures: list[list]):
    # Get unique Origin_seq values
    unique_origins = list(range(len(kde_figures)))
    
    # Create an empty figure
    fig = go.Figure()
    
    # Prepare dropdown menu options
    dropdown_buttons = []
    
    # Dictionary to map origin to trace indices
    origin_traces = {origin: [] for origin in unique_origins}

    for original_seq_number, figures in enumerate(kde_figures):
        visible = (original_seq_number == 0)
        for i, kde_fig in enumerate(figures):
            for trace in kde_fig['data']:
                trace.visible = visible
                trace.name = f"{original_seq_number}_{trace.name}"
                current_index = len(fig.data)
                origin_traces[original_seq_number].append(current_index)
                fig.add_trace(trace)

    
    # Create buttons for each origin
    for i, origin in enumerate(unique_origins):
        # Create visibility array - False for all traces by default
        visibility = [False] * len(fig.data)
        
        # Set visible=True ONLY for traces belonging to this origin
        for idx in origin_traces[origin]:
            visibility[idx] = True
            
        # Create button for dropdown menu
        button = dict(
            method="update",
            label=f"Origin_seq: {origin}",
            args=[{"visible": visibility},
                  {"title": f"KDE Distribution - Origin sequnce: {origin}"}]
        )
        
        dropdown_buttons.append(button)
    
    # Update layout with dropdown menu
    fig.update_layout(
        updatemenus=[
            dict(
                active=0,
                buttons=dropdown_buttons,
                pad={"r": 10, "t": 10},
                showactive=True,
                xanchor="left",
                y=1.08,
                yanchor="top"
            ),
        ],
        title="KDE Distribution",
        uirevision="constant",
        height=800,
        width=1000,
        xaxis_title="Index",
        yaxis_title="Density",
        showlegend=True,
        template='simple_white',
        legend=dict(font=dict(size=8))
    )
    
    return fig.show(renderer='jupyterlab')

In [18]:
def plotly_bar(snp_dataframe2):
    unique_origin = snp_dataframe2['Origin_seq'].unique()
    
    # Seaborn-like color palette
    colors = px.colors.qualitative.Set2
    
    # Create figure
    fig = go.Figure()
    
    # Add a trace for each origin
    for i, origin in enumerate(unique_origin):
        df_subset = snp_dataframe2[snp_dataframe2['Origin_seq'] == origin]
        
        # Create bar trace
        fig.add_trace(go.Bar(
            x=df_subset['Seq. No'],
            y=df_subset['number of mutations'],
            name=str(origin),
            marker_color=colors[i % len(colors)],
            visible = True,  # Only the first origin is visible by default
        ))
    

    # Update layout
    fig.update_layout(
        title="Bar plot of number of mutations",
        height=800,
        width=1000,
        xaxis_title="Sequence number",
        yaxis_title="Number of mutations",
        showlegend=True,
        template='simple_white',
        legend=dict(font=dict(size=8))
    )
    
    return fig.show(renderer='png')

In [8]:

def save_sequence_as_fasta(sequence: MutableSeq, filename: str, seq_id: str) -> None:
    seq_obj = Seq(str(sequence))
    record = SeqRecord(
        seq_obj, 
        id=seq_id,
        description=""
    ) 
    SeqIO.write(record, filename, "fasta")

In [9]:
def make_paths(number_of_seq: int, number_of_mut: int, directory: str) -> None:
    paths_dir = os.path.join(directory, "genome_paths")
    os.makedirs(paths_dir, exist_ok=True)
    with open(os.path.join(paths_dir, "origins.txt"), 'w') as f:
        for i in range(number_of_seq):
            path = os.path.join(directory, f"origin_{i}.fasta")
            f.write(path + '\n')

    for i in range(number_of_seq):
        with open(os.path.join(paths_dir, f"mutations_{i}.txt"), 'w') as f:
            for j in range(number_of_mut):
                path = os.path.join(directory, f"mutated_{i}_{j}.fasta")
                f.write(path + '\n')


In [21]:
# N=int(input('how many sequences: '))
N=10
# seq_length = int(input("Input the desired length for random sequence: "))
seq_length = 20_000

In [22]:
def perform_mutations(
        mutation_func: Callable[[MutableSeq, float, bool], tuple[MutableSeq, GenomeMutations]], 
        num_seq: int, 
        genome_length: int,
        mutation_rates: list[float], 
        psrandom: bool, 
        output_directory: str
    ) -> tuple[dict, list[GenomeMutations]]:
    
    to_plot = []
    os.makedirs(output_directory, exist_ok=True)
    for i, sequence in enumerate(make_synth_seq(num_seq, genome_length)):
        save_sequence_as_fasta(sequence, filename=f"{output_directory}/origin_{i}.fasta", seq_id=f"origin_{i}")
        print(i/num_seq*100,"%")
        curr_plot = []
        for j, mutation_rate in enumerate(mutation_rates):
            mutated_sequence, mutation_indexes = mutation_func(sequence, mutation_rate, psrandom)
            
            save_sequence_as_fasta(mutated_sequence, filename=f"{output_directory}/mutated_{i}_{j}.fasta", seq_id=f"mutated_{i}_{j}")
            plot_figure = generate_kde_fig(mutation_indexes, j, mutation_rate, color=plt.cm.tab20(i/len(mutation_rates)))

            curr_plot.append(plot_figure)
        to_plot.append(curr_plot)
    make_paths(num_seq, len(mutation_rates), output_directory)
    plotly_kde_dropdown(to_plot)

In [23]:
def insert_bases(sequence: MutableSeq, number_of_insertions: int, psrandom: bool) -> tuple[MutableSeq, GenomeMutations]:
    mutated_sequence = copy(sequence)
    if psrandom == True:
        # Select 3 random splits of the sequence indexes to mutate 
        index_range = np.arange(len(sequence))
        split_index = np.array_split(index_range, 10) # Split the indexes into 10 parts
        selected_index = random.sample(split_index,3) # Select 3 random splits
        joined_selected_index = np.concatenate(selected_index)
        indexes = np.random.choice(joined_selected_index,size=number_of_insertions, replace=False)
    else:
        indexes = np.random.choice(len(sequence),size=number_of_insertions, replace=False)
        
    mutation_indexes = GenomeMutations([], len(indexes))
    for i in sorted(indexes, reverse=True):
        mutation_indexes.indices.append(Mutation(i, "ins"))
        new_base = random.choice(basiupper)
        mutated_sequence.insert(i, new_base)
    return mutated_sequence, mutation_indexes

In [24]:
def delete_bases(sequence: MutableSeq, num_deletions: int, psrandom: bool) -> tuple[MutableSeq, GenomeMutations]:
    mutated_sequence = copy(sequence)
    if psrandom:
        # Select 3 random splits of the sequence indexes to delete
        index_range = np.arange(len(sequence))
        split_index = np.array_split(index_range, 10)  # Split the indexes into 10 parts
        selected_index = random.sample(split_index, 3)  # Select 3 random splits
        joined_selected_index = np.concatenate(selected_index)
        indices_to_delete = np.random.choice(joined_selected_index,size=num_deletions, replace=False)
    else:
        # Randomly select indices to delete
        indices_to_delete = np.random.choice(len(mutated_sequence),size=num_deletions, replace=False)
    mutation_indexes = GenomeMutations([], len(indices_to_delete))
    for index in sorted(indices_to_delete, reverse=True):
        mutation_indexes.indices.append(Mutation(index, "del"))
        del mutated_sequence[index]
    
    return mutated_sequence, mutation_indexes

In [25]:
def merge_mutations(deletions: GenomeMutations, insertions: GenomeMutations) -> GenomeMutations:
    n_total = deletions.number_of_mutations + insertions.number_of_mutations
    merged_mutations = GenomeMutations([None for _ in range(n_total)], n_total)
    # deletion_array= sorted(deletions.indices, key=lambda x: x.index)
    # insertion_array= sorted(insertions.indices, key=lambda x: x.index)
    deletion_array = deletions.indices
    insertion_array = insertions.indices
    del_ctr = 0
    ins_ctr = 0
    tot_ctr = 0

    while del_ctr < deletions.number_of_mutations or ins_ctr < insertions.number_of_mutations:
        if del_ctr == deletions.number_of_mutations:
            mutation = insertion_array[ins_ctr]
            ins_ctr += 1
        elif ins_ctr == insertions.number_of_mutations:
            mutation = deletion_array[del_ctr]
            del_ctr += 1
        elif deletion_array[del_ctr].index <= insertion_array[ins_ctr].index:
            mutation = deletion_array[del_ctr]
            del_ctr += 1
            for i in range(ins_ctr, len(insertion_array)):
                insertion_array[i].index += 1
        else:
            mutation = insertion_array[ins_ctr]
            ins_ctr += 1
        merged_mutations.indices[tot_ctr] = mutation
        tot_ctr += 1

    return merged_mutations

# def merge_mutations_da_finocchi(deletions: GenomeMutations, insertions: GenomeMutations) -> GenomeMutations:
#     return GenomeMutations(deletions.indices+insertions.indices, deletions.number_of_mutations + insertions.number_of_mutations)

def indels(sequence: MutableSeq, mutation_rate: float, psrandom: bool) -> tuple[MutableSeq, GenomeMutations]:
    num_mutation_adjustments = len(sequence) * mutation_rate**2 * 5/8
    num_deletions = int(np.random.binomial(len(sequence), mutation_rate/2) + num_mutation_adjustments)
    num_insertions = int(np.random.binomial(len(sequence), mutation_rate/2) + num_mutation_adjustments)
    
    mutated_sequence, deletions = delete_bases(sequence, num_deletions, psrandom)
    mutated_sequence, insertions = insert_bases(sequence, num_insertions, psrandom)
    # print(f"Mutated {len(deletions.indices)} deletions and {len(insertions.indices)} insertions")
    mutation_indexes = merge_mutations(deletions, insertions)
    return mutated_sequence, mutation_indexes

# Generating the mutated sequences for each method 
The plot shows the KDE distribution of the indexes where the mutation happend 

In [ ]:
to_plot = perform_mutations(
    mutation_func=snp_1_opt, 
    num_seq=N, 
    genome_length=seq_length,
    mutation_rates=mut_array, 
    psrandom=False, 
    output_directory="/home/lorenzo/Documents/tesi_2/Fastani-skani-comparison/SNP_random"
)


In [ ]:
to_plot = perform_mutations(
    mutation_func=indels, 
    num_seq=N, 
    genome_length=20_000,
    mutation_rates=mut_array, 
    psrandom=False, 
    output_directory="/home/lorenzo/Documents/tesi_2/Fastani-skani-comparison/INDEL_random"
)


In [ ]:
to_plot = perform_mutations(
    mutation_func=snp_1_opt, 
    num_seq=N, 
    genome_length=seq_length,
    mutation_rates=mut_array, 
    psrandom=True, 
    output_directory="/home/lorenzo/Documents/tesi_2/Fastani-skani-comparison/SNP_psrandom"
)


In [ ]:
to_plot = perform_mutations(
    mutation_func=indels, 
    num_seq=N, 
    genome_length=seq_length,
    mutation_rates=mut_array, 
    psrandom=True, 
    output_directory="/home/lorenzo/Documents/tesi_2/Fastani-skani-comparison/INDEL_psrandom"
)


# Comparing FastANI and skani 

In [26]:
# path= "\home\lorenzo\Documents\\tesi_2"
DIRECTORY1 = '\home\lorenzo\Documents\tesi_2\fastani_output'
DIRECTORY2 = '\home\lorenzo\Documents\tesi_2\skani_output'
os.makedirs(DIRECTORY1, exist_ok=True)
os.makedirs(DIRECTORY2, exist_ok=True)

In [10]:
@dataclass(slots=True)
class Times:
    best: list[float]
    worst: list[float]
    avg: list[float]
    stdv: list[float]

@dataclass(slots=True)
class AllRuns:
    run : list[Times]
    runN : int
    mutrate : list[float]


In [11]:
def run_method_ani(input_dir : str , output_dir : str, num_seq: int, method: str) -> AllRuns:
    fastANI_times = AllRuns([],0,0)
    skani_times = AllRuns([],0,0)
    reference_path = input_dir
    query_path = os.path.join(input_dir, "genome_paths")
    # method = method.lower()
    os.makedirs(output_dir, exist_ok=True)
    if method == "fastani":
        for i in range(num_seq):
            cmd = lambda: subprocess.run([
                'fastANI',
                "--ql",f"{query_path}/mutations_{i}.txt",
                "-r",f"{reference_path}/origin_{i}.fasta",
                "-o",f"{output_dir}/{method}_output_{i}.txt",
            ],stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            time = %timeit -n 1 -o cmd()
            fastANI_times.run.append(Times(time.best, time.worst, time.average, time.stdev))
            fastANI_times.runN += time.loops

    else:
        for i in range(num_seq):
            cmd = lambda: subprocess.run([
                method,
                "dist",
                "--ql",f"{query_path}/mutations_{i}.txt",
                "-r",f"{reference_path}/origin_{i}.fasta",
                "-o",f"{output_dir}/{method}_output_{i}.txt",
            ],stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            time = %timeit -n 1 -o cmd()
            skani_times.run.append(Times(time.best, time.worst, time.average, time.stdev))
            skani_times.runN += time.loops

            

In [45]:
def plot_timedependency_N(
        fastANItimes: AllRuns,
        skanitimes: AllRuns
        ):
    fast_avg = [x.avg for x in fastANItimes.run]
    skani_avg = [x.avg for x in skanitimes.run]
    fast_stdv = [x.stdv for x in fastANItimes.run]
    skani_stdv = [x.stdv for x in skanitimes.run]
    x = np.arange(len(fast_avg))

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x, 
        y=fast_avg, 
        mode='lines+markers', 
        name='FastANI',
        line=dict(color='blue'),
        error_y=dict(type='data', array=fast_stdv, visible=True)
    ))
    fig.add_trace(go.Scatter(
        x=x, 
        y=skani_avg, 
        mode='lines+markers', 
        name='Skani',
        line=dict(color='red'),
        error_y=dict(type='data', array=skani_stdv, visible=True)
    ))

    # regression = linregress(x, fast_avg)
    # slope = regression.slope
    # intercept = regression.intercept
    # x_fit = np.linspace(0, (len(fast_avg)+10)-1, (len(fast_avg)+10))
    # y_fit = slope * x_fit + intercept

    # fig.add_trace(go.Scatter(
    #     x=x_fit,
    #     y=y_fit,
    #     mode='lines',
    #     name='Regression Line',
    #     line=dict(color='green', dash='dash'),
    #     opacity=0.5,
    #     hovertemplate="y=%.2fx+%.2f<extra></extra><br>R :%.4f</br>" % (slope, intercept,regression.rvalue)
    # ))

    fig.add_trace(go.Bar(
        x=x, 
        y=fast_avg,
        name='FastANI',
        marker_color='blue',
        error_y=dict(type='data', array=fast_stdv, visible=True),
        visible=False
    ))
    fig.add_trace(go.Bar(
        x=x, 
        y=skani_avg,
        name='Skani',
        marker_color='red',
        error_y=dict(type='data', array=skani_stdv, visible=True),
        visible=False
    ))
    fig.update_traces(
        hovertemplate="Number of sequences: %{x}<br>Time: %{y:.2f} seconds<extra></extra><br>Standard deviation: %{error_y.array:.4f}"
    )
    updatemenus=[
        dict(
            type="buttons",
            buttons=[
                dict(label="Bar",
                     method="update",
                     args=[{"visible": [False, False, True, True]}]),
                dict(label="Line",
                     method="update",
                     args=[{"visible": [True, True, False, False]}])
            ],
            direction="right",
            showactive=True,
            pad={"r": 10, "t": 10},
            x=0,
            xanchor="left",
            y=1.08,
            yanchor="top"
        )
    ]
    fig.update_layout(
        updatemenus=updatemenus,
        
        title="Time Dependency on Number of Sequences",
        xaxis_title="Number of Sequences",
        yaxis_title="Time (seconds)",
        template='simple_white',
        legend=dict(font=dict(size=8)),
        hovermode='x',
        height=800,
        width=1500
    )

    fig.show(renderer='jupyterlab')

In [13]:
def plot_timedependency(timelist:AllRuns, method:str, xlabel:str):
    best = [x.best for x in timelist.run]
    worst = [x.worst for x in timelist.run]
    avg = [x.avg for x in timelist.run]
    stdv = [x.stdv for x in timelist.run]
    x= [i for i in range(timelist.runN+1)]
    mutrate = [x for x in timelist.mutrate]
    fig = go.Figure([])
    timesbyrate = defaultdict(list)
    for mut_rate, time in zip(mutrate, avg):
        timesbyrate[mut_rate].append(time)
    for mut in sorted(timesbyrate.keys()):
        fig.add_trace(go.Box(
            y= timesbyrate[mut],
            name = f'Mut rate: {mut.round(4)}',
            boxmean='sd',
            boxpoints='all',
            jitter=0.3
        ))
    updatemenus = [
    dict(
        type="buttons",
        direction="right",
        buttons=[
            dict(
                args=[{"boxpoints": "all"}],
                label="Show All Points",
                method="restyle"
            ),
            dict(
                args=[{"boxpoints": "outliers"}],
                label="Show Only Outliers",
                method="restyle"
            ),
            dict(
                args=[{"boxpoints": False}],
                label="Hide All Points",
                method="restyle"
            )
        ],
        pad={"r": 10, "t": 10},
        showactive=True,
        x=0,
        xanchor="left",
        y=1.08,
        yanchor="top"
    )
    ]
    fig.update_layout(
        title=f"{method} Time vs Mutation Rate",
        xaxis_title= xlabel,
        yaxis_title="Time (s)",
        height=800,
        width=1500,
        template='simple_white',
        legend=dict(font=dict(size=8)),
        hovermode="x",
        updatemenus=updatemenus,
        showlegend=True
    )
    fig.show(renderer='jupyterlab')

In [44]:
def methodANI_mut_dependency(
        N: int,
        N_pairwise: int,
        mut_func: Callable[[MutableSeq, float, bool], tuple[MutableSeq, GenomeMutations]],
        method: str,
        seq_length: int,
        mutation_array: list[float],
        output_path: str
) -> AllRuns:
    
    timelist = AllRuns([],0,[])
    if method == "skani":
        os.makedirs(os.path.join(output_path, "skani_genome_paths"), exist_ok=True)
        os.makedirs(os.path.join(output_path,"skani_output"), exist_ok=True)
        skani_output=os.path.join(output_path,"skani_output")
        paths_dir = os.path.join(output_path, "skani_genome_paths")
    else:
        os.makedirs(os.path.join(output_path, "fastANI_genome_paths"), exist_ok=True)
        os.makedirs(os.path.join(output_path,"fastANI_output"), exist_ok=True)
        paths_dir = os.path.join(output_path, "fastANI_genome_paths")
        fastani_output=os.path.join(output_path,"fastANI_output")
    
    os.makedirs(output_path, exist_ok=True)
    method = method.lower()
    threads = str(multiprocessing.cpu_count())

    for i,sequence in enumerate(make_synth_seq(N, seq_length)):
        reference_path = f"{output_path}/reference_{i}.fasta"
        save_sequence_as_fasta(sequence, filename=f"{output_path}/reference_{i}.fasta", seq_id=f"reference_{i}")
        
        query_paths = []
        for k,mut_rate in enumerate(mutation_array):
            query_list_path = os.path.join(paths_dir, f"queries_{k}.txt")
            for j in range(1,N_pairwise+1):
                mutated_sequence, mutation_indexes = mut_func(sequence,mut_rate , False)
                save_sequence_as_fasta(mutated_sequence, filename=f"{output_path}/query_{i}_{k}_{j-1}.fasta", seq_id=f"query_{i}_{k}_{j-1}_{mut_rate}")
                query_paths.append(f"{output_path}/query_{i}_{k}_{j-1}.fasta")
                
    # print(f'{query_paths[:N_pairwise]}')
    # start_idx = 0
    # for i in query_paths[start_idx:start_idx+N_pairwise]:
    #     path_file = os.path.join(paths_dir, f"paths_{start_idx}.txt")
    #     with open(path_file, 'w') as f:
    #         for path in query_paths[start_idx:start_idx+N_pairwise]:
    #             f.write(path + '\n')
    #     start_idx += N_pairwise
    
    # for  i in range(len(mutation_array)*N_pairwise):
    #     path_file = os.path.join(paths_dir, f"paths_{i}.txt")
    #     with open(path_file, 'w') as f:
    #         for path in query_paths[i]:
    #             f.write(path + '\n')
                
    if method == "fastani":
        for  i in range(len(mut_array)):
            for j in range(N_pairwise):
                cmd = lambda: subprocess.run([
                    'fastANI',
                    "-t", threads,
                    "-q", reference_path,
                    "-r", f"{output_path}/query_0_{i}_{j}.fasta",
                    "-o", f"{fastani_output}/0_{i}_{j}.txt"
                ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                fastani_time = %timeit -n 5 -o -q cmd()
                timelist.run.append(Times(fastani_time.best, fastani_time.worst, fastani_time.average, fastani_time.stdev))
                timelist.runN += 1
                timelist.mutrate.append(mut_array[i])
    else:
        for i in range(len(mut_array)):
            for j in range(N_pairwise):
                cmd = lambda: subprocess.run([
                    method,
                    "dist",
                    "-t", threads,
                    "-q", f"{output_path}/query_0_{i}_{j}.fasta",
                    "-r", reference_path,
                    "-o", f"{skani_output}/0_{i}_{j}.txt"
                ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                skani_time = %timeit -n 5 -o -q cmd()
                timelist.run.append(Times(skani_time.best, skani_time.worst, skani_time.average, skani_time.stdev))
                timelist.runN +=1
                timelist.mutrate.append(mut_array[i])
    #         print(i)
    for filename in os.listdir(output_path):
        if os.path.isfile(os.path.join(output_path, filename)):
            os.remove(os.path.join(output_path, filename))
    if method == "skani":
    
        for filename in os.listdir(skani_output):
            if os.path.isfile(os.path.join(skani_output, filename)):
                os.remove(os.path.join(skani_output, filename))
    else:
        for filename in os.listdir(fastani_output): 
            if os.path.isfile(os.path.join(fastani_output, filename)):
                os.remove(os.path.join(fastani_output, filename))
    # print([i for i in range(timelist.runN+1)])
    X_label: str = "Mutation Rate"
    plot_timedependency(timelist, method, X_label)

## ANI calculation time vs mutation rate 
Performs 1 vs 1 ANI calculation at various mutation rates 

In [34]:
timeit2 = methodANI_mut_dependency(
    N=1,
    N_pairwise=50,
    mut_func=snp_1_opt,
    method="skani",
    seq_length=5_000_000,
    mutation_array=mut_array,
    output_path="/home/lorenzo/Documents/tesi_2/fastANI-skani-comparison/skani-mut-dependency"
)

In [35]:
timeit2 = methodANI_mut_dependency(
    N=1,
    N_pairwise=50,
    mut_func=snp_1_opt,
    method="fastani",
    seq_length=5_000_000,
    mutation_array=mut_array,
    output_path="/home/lorenzo/Documents/tesi_2/fastANI-skani-comparison/fastANI-mut-dependency"
)

In [47]:
timeit2 = methodANI_mut_dependency(
    N=1,
    N_pairwise=50,
    mut_func=snp_1_opt,
    method="fastani",
    seq_length=5_000_000,
    mutation_array=mut_array,
    output_path="/home/lorenzo/Documents/tesi_2/fastANI-skani-comparison/fastANI-mut-dependency"
)

In [39]:
def methodANI_N_dependency(
        N: int,
        seq_length: int,
        number_of_pairwise: int,
        output_path: str
) -> AllRuns:
    fastANI_timelist = AllRuns([],0,[])
    skani_timelist = AllRuns([],0,[])
    # number_of_pairwise = np.linspace(0, number_of_pairwise-1, number_of_pairwise)

    # method.lower()
    

    # os.makedirs(os.path.join(output_path, "skani_genome_paths"), exist_ok=True)
    os.makedirs(os.path.join(output_path,"skani_output"), exist_ok=True)
    skani_output=os.path.join(output_path,"skani_output")
    # paths_dir = os.path.join(output_path, "skani_genome_paths")
    # os.makedirs(os.path.join(output_path, "fastANI_genome_paths"), exist_ok=True)
    os.makedirs(os.path.join(output_path,"fastANI_output"), exist_ok=True)
    # paths_dir = os.path.join(output_path, "fastANI_genome_paths")
    fastani_output=os.path.join(output_path,"fastANI_output")
    
    os.makedirs(os.path.join(output_path, "genome_paths"), exist_ok=True)
    paths_dir = os.path.join(output_path, "genome_paths")
    os.makedirs(output_path, exist_ok=True)

    threads = str(multiprocessing.cpu_count())

    for i,sequence in enumerate(make_synth_seq(N, seq_length)):
        reference_path = f"{output_path}/reference_{i}.fasta"
        save_sequence_as_fasta(sequence, filename=f"{output_path}/reference_{i}.fasta", seq_id=f"reference_{i}")
        
        query_paths = []
        for j in range(1,number_of_pairwise+1):
            query_list_path = os.path.join(paths_dir, f"queries_{i}.txt")
            
            # save_sequence_as_fasta(sequence, filename=f"{output_path}/query_{i}_{j-1}.fasta", seq_id=f"query_{i}_{j-1}")
            # query_paths.append(f"{output_path}/query_{i}_{j-1}.fasta")
            query_paths.append(reference_path)
            
    total_paths = min(number_of_pairwise,len(query_paths))
    for j in range(1, total_paths+1):
        path_file = os.path.join(paths_dir, f"paths_{j}.txt")
        with open(path_file, 'w') as f:
            for path in query_paths[:j]:
                f.write(path + '\n') 
                
    # print(f' {skani_output} \n {fastani_output}')

    
    for i in range(len(query_paths)):
        cmd = lambda: subprocess.run([
            'skani',
            "dist",
            "-t", threads,
            "-r",reference_path,
            "--ql", f'{paths_dir}/paths_{i+1}.txt',
            "-o", f"{skani_output}/{i}.txt"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        time1 = %timeit -r 3 -n 1 -o -q cmd()
        skani_timelist.run.append(Times(time1.best,time1.worst,time1.average,time1.stdev))
        skani_timelist.runN = i-1


    for i in range(len(query_paths)):
        cmd = lambda: subprocess.run([
            'fastANI',
            "-t", threads,
            "-q", reference_path,
            "--rl", f'{paths_dir}/paths_{i+1}.txt',
            "-o", f"{fastani_output}/{i}.txt"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        time2 = %timeit -r 3 -n 1 -o -q cmd()
        fastANI_timelist.run.append(Times(time2.best,time2.worst,time2.average,time2.stdev))
        fastANI_timelist.runN = j

    for filename in os.listdir(output_path):
        if os.path.isfile(os.path.join(output_path, filename)):
            os.remove(os.path.join(output_path, filename))
    for filename in os.listdir(paths_dir):
        if os.path.isfile(os.path.join(paths_dir, filename)):
            os.remove(os.path.join(paths_dir, filename))
            
    for filename in os.listdir(skani_output):
        if os.path.isfile(os.path.join(skani_output, filename)):
            os.remove(os.path.join(skani_output, filename))

    for filename in os.listdir(fastani_output): 
        if os.path.isfile(os.path.join(fastani_output, filename)):
            os.remove(os.path.join(fastani_output, filename))

    del query_list_path, query_paths
    plot_timedependency_N(fastANI_timelist,skani_timelist)

## ANI calculation time vs number of query sequences 
Performs 1 vs all ANI calculations with increasing number of queries 

In [46]:
timelist = methodANI_N_dependency(
    N=1,
    seq_length=5_000_000,
    number_of_pairwise=100,
    output_path='/home/lorenzo/Documents/tesi_2/fastANI-skani-comparison/N-time-dependency'
)


# ANI accuracy comparison 
ANI scores for each method are plotted against the theoretical identity and the ANI score computed with pyorthoANI - an alignment based method 

In [187]:
run_method_ani(
    input_dir= '/home/lorenzo/indels',
    output_dir= '/home/lorenzo/indels',
    num_seq= N,
    method= 'skani'
)

The slowest run took 4.25 times longer than the fastest. This could mean that an intermediate result is being cached.
5.2 ms ± 3.46 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
3.54 ms ± 581 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
3.62 ms ± 933 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
3.59 ms ± 590 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
3.5 ms ± 722 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
3.38 ms ± 598 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
3.4 ms ± 732 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
3.46 ms ± 677 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
3.45 ms ± 704 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
3.74 ms ± 708 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [188]:
run_method_ani(
    input_dir= '/home/lorenzo/indels',
    output_dir= '/home/lorenzo/indels',
    num_seq= N,
    method= 'fastani'
)

36.3 ms ± 836 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
35.5 ms ± 954 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
35 ms ± 480 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
36.5 ms ± 597 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
34.2 ms ± 309 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
34.4 ms ± 506 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
35.1 ms ± 1.31 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
34.4 ms ± 362 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
35.7 ms ± 297 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
34.2 ms ± 1.02 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [189]:
def make_fastani_df(path: str, num_of_origins: int) -> pd.DataFrame:
    df_list = []
    for i in range(num_of_origins):
        temp_df = pd.read_csv(f'{path}/fastani_output_{i}.txt', sep="\t", header=None)
        temp_df.columns = ['query', 'reference', 'fastANI', 'seq_fragments', 'orth_matches']
        df_list.append(temp_df)
        fastani_df = pd.concat(df_list, ignore_index=True)
    return fastani_df

def make_skani_df(path: str, num_of_origins: int) -> pd.DataFrame:
    df_list = []
    for i in range(num_of_origins):
        temp_df = pd.read_csv(f'{path}/skani_output_{i}.txt', sep="\t", header=None)
        temp_df.columns = temp_df.iloc[0]
        temp_df = temp_df[1:]
        df_list.append(temp_df)
        skani_df = pd.concat(df_list, ignore_index=True)
        
    cols = list(skani_df.columns)
    r,q = cols.index('Ref_file'), cols.index('Query_file')
    cols[r], cols[q] = cols[q], cols[r]
    reordered_skani_df = skani_df[cols]
    reordered_skani_df = reordered_skani_df.drop(columns=['Ref_name', 'Query_name'])
    reordered_skani_df = reordered_skani_df.rename(columns={'ANI':'skANI','Ref_file':'reference','Query_file':'query'})
    
    return reordered_skani_df

def make_merged_df(fastani_df: pd.DataFrame, skani_df: pd.DataFrame) -> pd.DataFrame:
    merged_df = pd.merge(fastani_df, skani_df, on=['query', 'reference'], how='outer')
    merged_df = merged_df.rename(columns={'query': 'Query', 'reference': 'Reference'})
    merged_df = merged_df[['Query', 'Reference', 'fastANI', 'skANI']]
    merged_df['fastANI']= merged_df['fastANI'].round(2)
    # Extract just the filename from the full paths
    merged_df['Query'] = merged_df['Query'].apply(os.path.basename)
    merged_df['Reference'] = merged_df['Reference'].apply(os.path.basename)
    global inverted_mut_array 
    inverted_mut_array = (1-mut_array)*100
    unique_seqs = merged_df['Query'].str.extract(r'mutated_\d+_(\d+)').astype(float)
    merged_df['predicted'] = unique_seqs.apply(lambda idx: inverted_mut_array[int(idx)], axis=1)

    return merged_df


In [190]:
fastani_df = make_fastani_df('/home/lorenzo/indels', N)
fastani_df

,query,reference,fastANI,seq_fragments,orth_matches
0,/home/lorenzo/indels/mutated_0_0.fasta,/home/lorenzo/indels/origin_0.fasta,99.8854,6,6
1,/home/lorenzo/indels/mutated_0_1.fasta,/home/lorenzo/indels/origin_0.fasta,99.6669,6,6
2,/home/lorenzo/indels/mutated_0_2.fasta,/home/lorenzo/indels/origin_0.fasta,99.3043,6,6
3,/home/lorenzo/indels/mutated_0_3.fasta,/home/lorenzo/indels/origin_0.fasta,98.8563,6,6
4,/home/lorenzo/indels/mutated_0_4.fasta,/home/lorenzo/indels/origin_0.fasta,98.3971,6,6
...,...,...,...,...,...
148,/home/lorenzo/indels/mutated_9_9.fasta,/home/lorenzo/indels/origin_9.fasta,94.6745,5,6
149,/home/lorenzo/indels/mutated_9_10.fasta,/home/lorenzo/indels/origin_9.fasta,93.0171,5,6
150,/home/lorenzo/indels/mutated_9_11.fasta,/home/lorenzo/indels/origin_9.fasta,90.7043,5,6
151,/home/lorenzo/indels/mutated_9_12.fasta,/home/lorenzo/indels/origin_9.fasta,86.7457,5,6


In [191]:
skani_df = make_skani_df('/home/lorenzo/indels', N)
skani_df

,query,reference,skANI,Align_fraction_ref,Align_fraction_query
0,/home/lorenzo/indels/mutated_0_0.fasta,/home/lorenzo/indels/origin_0.fasta,99.79,100.00,100.00
1,/home/lorenzo/indels/mutated_0_1.fasta,/home/lorenzo/indels/origin_0.fasta,99.53,100.00,100.00
2,/home/lorenzo/indels/mutated_0_10.fasta,/home/lorenzo/indels/origin_0.fasta,93.54,98.82,98.54
3,/home/lorenzo/indels/mutated_0_11.fasta,/home/lorenzo/indels/origin_0.fasta,91.69,91.78,91.72
4,/home/lorenzo/indels/mutated_0_12.fasta,/home/lorenzo/indels/origin_0.fasta,87.63,78.93,78.99
...,...,...,...,...,...
132,/home/lorenzo/indels/mutated_9_5.fasta,/home/lorenzo/indels/origin_9.fasta,97.71,99.21,99.20
133,/home/lorenzo/indels/mutated_9_6.fasta,/home/lorenzo/indels/origin_9.fasta,97.38,99.21,99.20
134,/home/lorenzo/indels/mutated_9_7.fasta,/home/lorenzo/indels/origin_9.fasta,96.65,99.21,99.21
135,/home/lorenzo/indels/mutated_9_8.fasta,/home/lorenzo/indels/origin_9.fasta,96.52,99.21,99.16


In [192]:
combined_ANI_df = make_merged_df(fastani_df, skani_df)
combined_ANI_df

/tmp/ipykernel_6262/439551514.py:39: FutureWarning:

Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead



,Query,Reference,fastANI,skANI,predicted
0,mutated_0_0.fasta,origin_0.fasta,99.89,99.79,99.9
1,mutated_0_1.fasta,origin_0.fasta,99.67,99.53,99.8
2,mutated_0_2.fasta,origin_0.fasta,99.30,99.16,99.7
3,mutated_0_3.fasta,origin_0.fasta,98.86,98.82,99.6
4,mutated_0_4.fasta,origin_0.fasta,98.40,98.30,99.5
...,...,...,...,...,...
148,mutated_9_9.fasta,origin_9.fasta,94.67,95.87,99.0
149,mutated_9_10.fasta,origin_9.fasta,93.02,93.71,98.0
150,mutated_9_11.fasta,origin_9.fasta,90.70,91.67,97.0
151,mutated_9_12.fasta,origin_9.fasta,86.75,86.12,96.0


In [193]:
@dataclass(slots=True)
class OrthoANI:
    Query: str
    Reference: str
    orthoANI: float

def orthoani_df( 
    ANI_touples: list[tuple[str, str]], 
    directory: str
) -> pd.DataFrame:
    
    orthoani_list = []
    for gen1, gen2 in ANI_touples:
        orthoani = pyorthoani.orthoani(
            read(os.path.join(directory, gen1), 'fasta'), 
            read(os.path.join(directory, gen2), 'fasta')
        ) * 100
        orthoani_list.append(OrthoANI(gen1, gen2, round(orthoani, 2)))
        
    return pd.DataFrame(orthoani_list)

In [194]:
combo_df = orthoani_df(
    ANI_touples= list(zip(combined_ANI_df['Query'], combined_ANI_df['Reference'])),
    directory='/home/lorenzo/bucio_de_culo'
    )
combo_df

,Query,Reference,orthoANI
0,mutated_0_0.fasta,origin_0.fasta,99.91
1,mutated_0_1.fasta,origin_0.fasta,99.81
2,mutated_0_2.fasta,origin_0.fasta,99.71
3,mutated_0_3.fasta,origin_0.fasta,99.68
4,mutated_0_4.fasta,origin_0.fasta,99.53
...,...,...,...
148,mutated_9_9.fasta,origin_9.fasta,99.00
149,mutated_9_10.fasta,origin_9.fasta,97.88
150,mutated_9_11.fasta,origin_9.fasta,96.90
151,mutated_9_12.fasta,origin_9.fasta,95.91


In [195]:
combo_ANI_ortho_df = pd.merge(combo_df,combined_ANI_df,on=['Query','Reference'],how='outer')
# combo_ANI_ortho_df.fillna(0, inplace=True)
combo_ANI_ortho_df


,Query,Reference,orthoANI,fastANI,skANI,predicted
0,mutated_0_0.fasta,origin_0.fasta,99.91,99.89,99.79,99.9
1,mutated_0_1.fasta,origin_0.fasta,99.81,99.67,99.53,99.8
2,mutated_0_2.fasta,origin_0.fasta,99.71,99.30,99.16,99.7
3,mutated_0_3.fasta,origin_0.fasta,99.68,98.86,98.82,99.6
4,mutated_0_4.fasta,origin_0.fasta,99.53,98.40,98.30,99.5
...,...,...,...,...,...,...
148,mutated_9_9.fasta,origin_9.fasta,99.00,94.67,95.87,99.0
149,mutated_9_10.fasta,origin_9.fasta,97.88,93.02,93.71,98.0
150,mutated_9_11.fasta,origin_9.fasta,96.90,90.70,91.67,97.0
151,mutated_9_12.fasta,origin_9.fasta,95.91,86.75,86.12,96.0


In [197]:
fig = go.Figure()

minvalue = combo_ANI_ortho_df['orthoANI'].min()
usabley = combo_ANI_ortho_df[combo_ANI_ortho_df['skANI']!='nan']['skANI']

# Create traces for orthoANI x-axis (set 1)
# skANI trace with orthoANI as x
fig.add_trace(go.Scattergl(
    name='skANI (vs orthoANI)',
    x=combo_ANI_ortho_df['orthoANI'],
    y=usabley.astype(float),
    mode='markers',
    marker=dict(symbol='circle-open', size=10, color='red', opacity=1),
    text= combo_ANI_ortho_df['Query'] + ' vs ' + combo_ANI_ortho_df['Reference'] +
         '<br>orthoANI: ' + combo_ANI_ortho_df['orthoANI'].round(2).astype(str) +
         '<br>skANI: ' + combo_ANI_ortho_df['skANI'].astype(str),
    hoverinfo='text',
    visible=True
))

# fastANI trace with orthoANI as x
fig.add_trace(go.Scattergl(
    name='fastANI (vs orthoANI)',
    x=combo_ANI_ortho_df['orthoANI'],
    y=combo_ANI_ortho_df['fastANI'],
    mode='markers',
    marker=dict(symbol='diamond-open', size=10, color='blue', opacity=1, line=dict(width=1,color='red')),
    text=combo_ANI_ortho_df['Query'] + ' vs ' + combo_ANI_ortho_df['Reference'] +
         '<br>orthoANI: ' + combo_ANI_ortho_df['orthoANI'].round(2).astype(str) +
         '<br>fastANI: ' + combo_ANI_ortho_df['fastANI'].round(2).astype(str),
    hoverinfo='text',
    visible=True
))

# Add bisector line for orthoANI
fig.add_trace(go.Scattergl(
    name='y=x (orthoANI)',
    x=[minvalue, 100],
    y=[minvalue, 100],
    mode='lines',
    line=dict(color='black', width=2, dash='dash'),
    hoverinfo='none',
    visible=True
))



fig.add_trace(go.Scattergl(
    name='skANI (vs mutation rate)',
    x=combo_ANI_ortho_df['predicted'],
    y=usabley.astype(float),
    mode='markers',
    marker=dict(symbol='circle-open', size=10, color='red', opacity=1),
    text= combo_ANI_ortho_df['Query'] + ' vs ' + combo_ANI_ortho_df['Reference'] +
         '<br>Mutation rate: ' + combo_ANI_ortho_df['predicted'].astype(str) +
         '<br>skANI: ' + combo_ANI_ortho_df['skANI'].astype(str),
    hoverinfo='text',
    visible=False
))

# fastANI trace with mutarray as x
fig.add_trace(go.Scattergl(
    name='fastANI (vs mutation rate)',
    # x=[inverted_mut_array[idx] if idx is not None and idx < len(inverted_mut_array) else None 
    #    for idx in combo_ANI_ortho_df['mut_idx']],
    x= combo_ANI_ortho_df['predicted'],
    y=combo_ANI_ortho_df['fastANI'],
    mode='markers',
    marker=dict(symbol='diamond-open', size=10, color='blue', opacity=1, line=dict(width=1,color='red')),
    text=combo_ANI_ortho_df['Query'] + ' vs ' + combo_ANI_ortho_df['Reference'] +
         '<br>Mutation rate: ' + combo_ANI_ortho_df['predicted'].astype(str)+
         '<br>fastANI: ' + combo_ANI_ortho_df['fastANI'].round(2).astype(str),
    hoverinfo='text',
    visible=False
))

# Add trend line for mutation rates
fig.add_trace(go.Scattergl(
    name='y=x (mutation rate)',
    x=[min(inverted_mut_array), 100],
    y=[min(inverted_mut_array), 100],
    mode='lines',
    line=dict(color='black', width=2, dash='dash'),
    hoverinfo='none',
    visible=False
))

# Add dropdown menu
fig.update_layout(
    updatemenus=[
        dict(
            buttons=list([
                dict(
                    args=[{'visible': [True, True, True, False, False, False]},
                          {'xaxis.title': 'orthoANI',
                           'title': 'Comparison of orthoANI with fastANI and skANI'}],
                    label="X-Axis: orthoANI",
                    method="update"
                ),
                dict(
                    args=[{'visible': [False, False, False, True, True, True]},
                          {'xaxis.title': 'Mutation Rate',
                           'title': 'Comparison of Mutation Rate with fastANI and skANI'}],
                    label="X-Axis: Mutation Rate",
                    method="update"
                )
            ]),
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0,
            xanchor="left",
            y=1.12,
            yanchor="top"
        ),
    ],
    title='Comparison of orthoANI with fastANI and skANI',
    xaxis_title='orthoANI',
    yaxis_title='fastANI / skANI',
    showlegend=True,
    template='simple_white',
    hovermode='closest',
    width=800,
    height=600
)

fig.show(renderer='jupyterlab')